# Full QSE workflow on H2

This notebook demonstrates a compact end-to-end QSE workflow in `qibochem`. We build an H2 molecule from inline coordinates, run a UCCSD VQE reference calculation, project the Hamiltonian into a QSE subspace, project the total-spin operator S^2 into the same style of subspace, and finally solve the generalized eigenvalue problem.

## Imports

Only the core workflow objects are needed: a molecule driver, a UCC ansatz, a statevector protocol, and the QSE utilities.

In [9]:
import numpy as np

from qibochem.driver import Molecule
from qibochem.ansatz.ucc import Ansatz_UCCSD
from qibochem.measurement.protocol import StateVectorProtocol
from qibochem.selected_ci import (
    QSE_Computable,
    generate_singlet_singles,
    solve_generalised_eigeneqn,
)

## 1. Build the molecule and run PySCF

The coordinates are defined directly in the notebook. For this minimal H2 example we use the full STO-3G molecular orbital space, so no active-space embedding is needed.

In [10]:
# Inline H2 geometry in Angstrom.
h2 = Molecule([
    ("H", (0.0, 0.0, 0.0)),
    ("H", (0.0, 0.0, 0.735)),
], basis="sto-3g")

h2.run_pyscf()

print(f"HF energy: {h2.e_hf:.12f} Ha")
print(f"Electrons: {h2.nelec}")
print(f"Spin orbitals: {h2.nso}")

HF energy: -1.116998996754 Ha
Electrons: 2
Spin orbitals: 4


## 2. Build the UCCSD VQE circuit

`Ansatz_UCCSD` builds both the parameter map and the initial circuit. `StateVectorProtocol` is used for exact expectation values during this tutorial.

In [11]:
protocol = StateVectorProtocol()
ansatz = Ansatz_UCCSD(h2, ferm_qubit_map="jw")

print(f"Ansatz parameters: {len(ansatz.param_names)}")
print(f"Initial circuit gates: {len(ansatz.circuit.queue)}")

Ansatz parameters: 3
Initial circuit gates: 198


## 2b. Run VQE

The fast VQE path evaluates the same ansatz using direct statevector Pauli rotations internally. The returned `final_circuit` remains a normal Qibo circuit that can be passed to QSE.

In [12]:
vqe_energy, vqe_params, final_circuit = ansatz.run_vqe(
    protocol,
    method="L-BFGS-B",
    fast=True,
)

print(f"VQE energy: {vqe_energy:.12f} Ha")
print("Optimized parameters:")
print(vqe_params)

VQE energy: -1.137306035753 Ha
Optimized parameters:
{'s0': np.float64(-8.213539269144507e-09), 's1': np.float64(-8.213539269144507e-09), 'd0': np.float64(0.11176849986109837)}


## 3. Build a Hamiltonian QSE computable

`QSE_Computable` projects an observable into the subspace generated by the chosen excitation generator. Passing `observable=None` uses the molecular Hamiltonian by default.

In [13]:
qse = QSE_Computable(
    molecule=h2,
    excitation_generator=generate_singlet_singles,
    observable=None,
    ferm_qubit_map="jw",
    map_threshold=1e-12,
)

## 3b. Run QSE and inspect H and S

The QSE run evaluates the projected Hamiltonian matrix `H` and the projected overlap matrix `S` on the VQE reference circuit. We then solve `H c = E S c` to get the QSE-corrected energies.

In [18]:
H, S = qse.run_qse(final_circuit, protocol)

print("H:")
print(np.real_if_close(H))
print("S:")
print(np.real_if_close(S))

qse_energies, qse_vectors, qse_rank = solve_generalised_eigeneqn(
    H,
    S,
    threshold=1e-8,
)

print()
print(f"Retained QSE rank: {qse_rank}")
print("QSE energies:")
print(qse_energies)
print(f"Corrected ground-state energy: {qse_energies[0]:.12f} Ha")
print(f"QSE correction from VQE: {qse_energies[0] - vqe_energy:.12e} Ha")

H:
[[-4.41241293e+00  3.61708812e-08 -3.25037479e-09 -8.02176567e-02]
 [ 3.61708812e-08 -4.04938503e-03  3.60791195e-02  3.64809873e-10]
 [-3.25037479e-09  3.60791195e-02 -3.21456927e-01  4.39656254e-09]
 [-8.02176567e-02  3.64809873e-10  4.39656254e-09  2.36240997e-02]]
S:
[[ 3.95023894e+00 -3.08169505e-08 -1.63245801e-08  1.22124533e-14]
 [-3.08169505e-08  2.48805316e-02 -2.21680000e-01  1.83220963e-09]
 [-1.63245801e-08 -2.21680000e-01  1.97511947e+00 -1.26601608e-08]
 [ 1.22124533e-14  1.83220963e-09 -1.26601608e-08  4.97610632e-02]]

Retained QSE rank: 3
QSE energies:
[-1.13730604 -0.16275316  0.49505774]
Corrected ground-state energy: -1.137306035753 Ha
QSE correction from VQE: 1.842970220878e-14 Ha


## 4. Build a spin QSE computable

The same QSE machinery can project other fermionic observables. Here we build a new computable for the total-spin operator S^2.

In [15]:
spin_qse = QSE_Computable(
    molecule=h2,
    excitation_generator=generate_singlet_singles,
    observable=h2.s2_operator(),
    ferm_qubit_map="jw",
    map_threshold=1e-12,
)

## 4b. Run spin QSE, inspect, and solve for spin values

The spin computable returns a projected S^2 matrix and its overlap matrix. Solving `S2 c = s(s+1) S c` gives the spin values represented in the same QSE subspace.

In [19]:
S2_matrix, spin_overlap = spin_qse.run_qse(final_circuit, protocol)

print("Projected S2 matrix:")
print(np.real_if_close(S2_matrix))

spin_values, spin_vectors, spin_rank = solve_generalised_eigeneqn(
    S2_matrix,
    spin_overlap,
    threshold=1e-8,
)

print()
print(f"Retained spin-QSE rank: {spin_rank}")
print("Projected S2 values:")
print(spin_values)

Projected S2 matrix:
[[ 1.07414078e-14 -8.27180613e-25 -8.27180613e-25  9.18709553e-15]
 [-8.27180613e-25  2.24820162e-15  0.00000000e+00  0.00000000e+00]
 [-8.27180613e-25  0.00000000e+00  2.19269047e-15 -8.27180613e-25]
 [ 9.18709553e-15  0.00000000e+00 -8.27180613e-25  1.09634524e-14]]

Retained spin-QSE rank: 3
Projected S2 values:
[7.63518982e-16 1.09669052e-15 2.22277568e-13]
